# Simulation-based Reinforcement Learning 

Previous modules provide building blocks for simulation-based reinforcement learning. Simulation, especially differentiable ones, can accelerate intelligent agent development without taking real-world risks. The most straightforward way of applying simulation to train reinforcement learning policy is to solve policy optimization in the optimal control framework:

$$
\label{eq-policyopt}
\begin{aligned}
& \arg\max\limits_{\theta} \mathbb{E}_{\mathbf{x}_0 \sim p_0(\mathbf{x})} [\int_{t=0}^T r(\mathbf{x}(t), \pi_{\theta}(\mathbf{x}(t)))dt ]\\
\text{s.t.} \quad & \mathbf{\dot{x}}(t) = f(\mathbf{x}(t), \pi_{\theta}(\mathbf{x}(t)))  \quad \forall t \in [0, T]   \\
\end{aligned}
$$

This usually yields a highly nonlinear optimization problem. Another potential issue is that, if we stick to deterministic policy and dynamics, the solution could be brittle given the policy does not sufficiently explore the state space during learning: $\pi_{\theta}(\mathbf{x}(t))$ may not work for $\mathbf{x}$ not encountered in the rollouts starting from $p_0(\mathbf{x})$. 

One way to effectively utilize simulation-based optimization is to take it as an oracle to label the action that an optimal policy should output. Specifically, we can iterate over the following process:
* Run a rollout with a behaviour policy (could be a probabilistic $\pi_{\theta}(\mathbf{x})$) to collect a few rollouts $\mathcal{R}$. 
* For each state $\mathbf{x}$ in the rollouts, invoke trajectory optimization to find the action to take i.e. $\mathbf{u}^* = \text{TrajOpt}(x)(0)$. This is often much more tractable than a direct policy optimization. 
* Train the target policy to make a prediction similar to the trajectory optimization solution, e.g. $\arg\min\limits_{\theta} \mathbb{E}_{\mathbf{x} \sim \mathcal{R}}[\|\pi_{\theta}(\mathbf{x}) - \mathbf{u}^*\|]$.

The most significant benefit of doing this is dealing with reinforcement learning as a regression problem. This is way simpler from the machine learning perspective. The flexibility of choosing the exploration strategy facilitates learning $\pi_{\theta}(\mathbf{x})$ with a better state space coverage. Note that we may not necessarily need an exhaustive trajectory optimization: a suboptimal $\mathbf{u}^*$ after one or a few gradient steps can still be valuable for guiding the gradual evolution of $\pi_{\theta}(\mathbf{x})$ towards the optimal behaviour. The class of methods is hence called _Guided Policy Search_ [@gps2016levine] and has been extensively applied in learning robot task policies. 

Simulation allows accessing any physical state in the virtual world. The white-box dynamic constraints about these states can be naturally leveraged in trajectory optimization. However, real systems may not have sensors to provide direct feedback about these states. 

For a concrete example, the robot simulation in below tracks the exact pose of all bodies  including the rubber duck to pick. The real robot does not have such a luxury and resorts to some sensors, e.g. a camera, for some indirect evidence about the duck's state. Training with the full physical state $\pi_{\theta}(\mathbf{x})$ would definitely be easier since the policy will be fed with the target location to reach. However such a policy is incompatible with the hardware to deploy, which expects $\pi_{\theta}(\mathbf{o})$ with $\mathbf{o}$ as an observation, e.g. a third-person-view or ego-camera image, caused by the underlying state $\mathbf{x}$ i.e. $p(\mathbf{o} | \mathbf{x})$. The regression idea can be used here to train the policy working with the partial observation to emulate the one that can decide upon the full simulation state, or say the _priviledged information_. This paradigm shares similarities to model _distillation_ where the target policy extracts the information from a more powerful policy, and also to the _teacher-student framework_ where a student policy is trained to imitate a teacher policy's prediction. 

```{figure} ../images/simvsreal.png
---
name: simvsreal
---
Simulation v.s. real policies: sensory input, embodiments and physical processes.  
```

:::{note}
Reconstructing the state $\mathbf{x}$ from $\mathbf{o}$ by inverting the process of $p(\mathbf{o}|\mathbf{x})$ can also yield a representation compatible to $\pi_{\theta}(\mathbf{x})$. This is often tackled as a state estimation task, e.g. running some computer vision algorithms or models to estimate the duck pose from an image. Such a state estimator can be learned by predicting the physical state in simulation. Hence the policy can take the raw input in deployment and work in an _end-to-end_ manner. 
:::


[](#simvsreal) implies other obstacles that might hinder the transfer of policies trained in simulation to reality. The sensory readings may not match closely, e.g. for real and rendered images. The target robot may face mismatch on parameters or even has entirely different embodiments. In all, the simulation is after all a model of the real world. The so-called _Sim-Real Gap_ is everywhere. Bridging the gap for complicated tasks pertains to state-of-the-art research. 

## Sim-to-Real: Domain Randomization

Let's take a look at the three-link robot example again. We create a gym environment by setting one model parameter, the link density, to $2700$. This is a _nominal_ model in that the reality might differ. 

In [1]:
import os
os.environ.pop('SIRL_USE_JAX', None)
from articulated_dynamics.math_utils import nplib, rotate
from articulated_dynamics.dynamics import Dynamics, Kinematics
from articulated_dynamics.robots import ThreeLinks
from articulated_dynamics.integrator import integrate_euler

import numpy as np
import gymnasium as gym

class MyThreeLinksEnv(gym.Env):
    def __init__(self, link_len=0.3, link_den=2700) -> None:
        super().__init__()

        #define the state and action space
        #Box indicate a space defined by intervals with low/high bounds. Could be unbounded as well.
        self.observation_space=gym.spaces.Box(low=-np.inf, high=np.inf, shape=(6,))   
        #the continuous action space is recommended to have a range [-1, 1] for Gaussian noised NN output
        #see discussion in https://github.com/hill-a/stable-baselines/issues/678
        self.action_space=gym.spaces.Box(low=-30, high=30, shape=(3,))

        self.model = ThreeLinks(link_len=link_len, link_den=link_den)
        self.link_len = link_len
        self.joint_lim = [-7*nplib.pi/8, 7*nplib.pi/8]
        
        self.state = None
        self.t = 0
        self.horizon = 200

        self.goal = nplib.array([0.5, 0.5, 0])

    def reset(self, seed=None):
        super().reset(seed=seed)
        #reset the environment: sampling a new initial state according to /pho; and reset the clock

        self.state = np.zeros(6)
        #only randomize the angular position of links
        self.state[0] = 0.  #self.np_random.uniform(low=-np.pi/6, high=np.pi/6)
        self.state[1] = 0.  #self.np_random.uniform(low=-np.pi/6, high=np.pi/6)
        self.state[2] = 0.  #self.np_random.uniform(low=-np.pi/6, high=np.pi/6)

        self.t = 0
        self.dt = 0.01
        return self.get_obs(), {}
    
    def get_obs(self):
        return self.state #nplib.concatenate((self.state, self.calc_tippos(self.state)))
    
    def calc_tippos(self, state):
        x_pose = Kinematics._forward_pos(self.model, state[:3])[0][-1]
        #get tip position
        xp_tip = x_pose.trans + rotate(nplib.array([0, self.link_len/2, 0]), x_pose.rot)
        return xp_tip

    def reward(self, s, a):
        #try to reach a goal
        xp_tip = self.calc_tippos(s)
        return -nplib.linalg.norm(xp_tip - self.goal) - nplib.linalg.norm(a) * 1e-5 - nplib.linalg.norm(s[3:]) * 5e-4
    
    def step(self, action):
        #evaluate the reward
        r = self.reward(self.state, action)

        q = nplib.array(self.state[:3])
        qd = nplib.array(self.state[3:])

        tau = action 

        qdd = Dynamics.forward(self.model, q, qd, tau)

        #integrate for a small time step, 0.01s here, for the next state
        q_next, qd_next = integrate_euler(self.model, self.dt, q, qd, qdd, symplectic=False)

        #apply joint limits
        qd_next[q_next > self.joint_lim[1]] = 0.
        qd_next[q_next < self.joint_lim[0]] = 0.
        qd_next[qd_next > 5] = 5.
        qd_next[qd_next < -5] = -5.
        q_next[q_next > self.joint_lim[1]] = self.joint_lim[1]
        q_next[q_next < self.joint_lim[0]] = self.joint_lim[0]

        #type conversion to numpy state format
        self.state = np.concatenate([np.array(q_next), np.array(qd_next)])

        #tick the time step and check if the rollout has reached the end
        self.t += 1
        terminated = ( self.t >= self.horizon )
        #depending the gym version, should return the finishing signal as solely a Done or more concrete terminated or truncated
        #see https://farama.org/Gymnasium-Terminated-Truncated-Step-API
        #the last dictionary allows passing some extra info when needed
        return self.get_obs(), r, terminated, False, {}

In [2]:
from stable_baselines3 import SAC
import articulated_dynamics.visualizer as viz

nominal_env = MyThreeLinksEnv(link_len=0.3, link_den=2700)

def rollout_from_model_policy(env, model):
    obs, _ = env.reset(seed=0)
    state_traj = []
    ret = 0
    for i in range(env.horizon):
        action, _ = model.policy.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = env.step(action)
        state_next = env.state
        state_traj.append(state_next)
        ret+=r
    return ret, state_traj

def animate_pos_traj(env, state_traj):
    viewer = viz.P3JSViewer(width=400, height=300)
    viewer.create_shapes(env.model)
    pos_traj = nplib.array(state_traj)[:, :3]
    viewer.place_shapes_qpos(env.model, pos_traj[0])
    viewer.place_marker('goal', env.goal, scale=0.05)

    viewer.show()
    anim = viewer.animate_qpos_traj(env.model, pos_traj, env.dt)
    return anim

We can train a policy to solve the simulation environment with standard RL algorithms, e.g. SAC, with a code snippet like below:

```python
model = SAC("MlpPolicy", nominal_env, verbose=1, seed=0)
model.learn(total_timesteps=80000)

model.save("threelinks_reacher")
```

Let's try to deploy the trained policy to another environment, e.g. now playing the role of the real physical environment, whose link density is actually $4000$:

In [3]:
real_env = MyThreeLinksEnv(link_len=0.3, link_den=4000)

#env = nominal_env
env = real_env
model = SAC.load("../data/threelinks_reacher")

ret, state_traj = rollout_from_model_policy(env, model)
anim = animate_pos_traj(env, state_traj) 
anim


Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

AnimationAction(clip=AnimationClip(duration=2.0, tracks=(VectorKeyframeTrack(name='scene/link_0.position', tim…

Unfortunately the policy fails. The Sim-Real gap is non-negligible in this case so a model trained on data generated from the nominal environment cannot work for a shifted environment. As hinted in [the previous module](../1_RLSystem/notebooks.ipynb), making a machine learning model trained on one distribution of data work well on out-of-distribution data is hard, and often tackled as a transfer learning/domain adaptation problem. In reinforcement learning, it is popular to use a paradigm called _domain randomization_ [@domarand2017iros]. The general idea is to exploit the flexibility of generating simulation data under a range of parameters, and hopefully a policy trained to work well on all these simulation environments will work better for the target environment, assuming the target environment is better covered by the parameter range. Formally, it is like an extension of the standard RL/Optimal Control problem:

$$
\label{eq-domarandform}
\begin{aligned}
& \arg\max\limits_{\theta} \mathbb{E}_{\mathbf{x}(0) \sim p_0(\mathbf{x}), \mathbf{w} \sim p_{sim}(\mathbf{w})} [ \int_{t=0}^T r(\mathbf{x}(t), \pi_{\theta}(\mathbf{x}(t)))dt ] \\
\text{s.t.} \quad & \dot{\mathbf{x}}(t) = f(\mathbf{x}(t), \pi_{\theta}(\mathbf{x}(t)), \mathbf{w})  \quad \forall t \in [0, T]   \\
\end{aligned}
$$

Here the extra $p_{sim}(\mathbf{w})$ denotes the range of the possible simulation parameters, such as the material density. The formulation draws a clear parallel to stochastic or robust optimal control so it is not hard to see why this can help on overcoming the sim-to-real gap. 

Implemention is straightforward and can leverage parallel computing since we want to evaluate a batch of simulation independent from each other. Many RL libraries provide wrappers to do this:

```python
from stable_baselines3.common.vec_env import SubprocVecEnv

envs = SubprocVecEnv( [ lambda: MyThreeLinksEnv(link_len=0.3, link_den=2700 + (nplib.random.rand()-0.5)*3000) for _ in range(20) ] )

model = SAC("MlpPolicy", envs, verbose=1, seed=0)
model.learn(total_timesteps=500000)

model.save("threelinks_reacher_dr")
```

And we can see that the domain randomized policy works better in the target environment.

In [4]:
model_dr = SAC.load("../data/threelinks_reacher_dr")

ret, state_traj = rollout_from_model_policy(env, model_dr)
anim = animate_pos_traj(env, state_traj) 
anim

Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

AnimationAction(clip=AnimationClip(duration=2.0, tracks=(VectorKeyframeTrack(name='scene/link_0.position', tim…

In general domain randomization may include many parameters such as link geometry, mass, friction coefficients, disturbance forces, control parameters and latency. Selecting which parameters and what range to randomize is subjective, although some [rules of thumb](https://github.com/isaac-sim/OmniIsaacGymEnvs/blob/main/docs/framework/domain_randomization.md) are consensus among practitioners. Domain randomization is currently the workhorse method to apply a policy attained from simulation-based reinforcement learning. The successful cases include skillful motion control for systems as complex as [quadrupedal](https://www.youtube.com/watch?v=8ClYBtfhkaw) and [humanoid](https://la.disneyresearch.com/publication/robot-motion-diffusion-model-motion-generation-for-robotic-characters/) robots. 

## Real-to-Sim: System Identification

To shrink the sim-to-real gap, it would be reasonable to find simulation parameters that can yield state trajectories with a better reality alignment. This requires to collect a few rollouts $\zeta_{real} = \{\mathbf{x}_0, \mathbf{u}_0, \mathbf{x}_1, \mathbf{u}_1, ... \}$ on the real system. The simulation parameters $\mathbf{p}$ can be found by solving:

$$
\label{eq-sysiden}
\begin{aligned}
& \arg\min\limits_{\mathbf{p}} \mathbb{E}_{\zeta \sim \zeta_{real}}[ \sum\limits_{t=0}^T \| \hat{\mathbf{x}}_t - \mathbf{x} \| ]\\
\text{s.t.} \quad & \hat{\mathbf{x}}_{t+1} = f(\hat{\mathbf{x}}_t, \mathbf{u}_t, \mathbf{p})  \quad t = 0, 1, ..., T-1   \\
\end{aligned}
$$

Note that this shares an identical formulation as the optimal control problem. Mathematically speaking, policy parameter $\theta$ and simulation parameter $\mathbf{p}$ are indistinguishable to a solver given both parameterize how the state trajectories evolve. 

Imagine a simple example about the three-link robot: we now mount an extra link (often referred as end-effector) such as a gripper to the last link but have no idea how long this extends the tip position. We may however measure the new tip position with e.g. computer vision techniques and collect some data about it under certain motion commands. This can be done in an offline process. The data can be used to estimate this unknown value which is subsequently used without needing the measurement anymore, see [](#paramest) .

```{figure} ../images/paramest.png
---
height: 300px
name: paramest
---
Unknown parameters e.g. the displacement of gripping point from the last link tip can be estimated from real data e.g. the tracked location of a gripped object (left). The predicted object locations (transparent dash lines on the right) are compared to the real data to gauge which offset guesses yield better alignment. 
```

In [7]:
class MyThreeLinksPlusEndEffectorEnv(MyThreeLinksEnv):
    offset = nplib.array([0, 0.1, 0])   #this is the assumed real offset that needs to be recovered

    def calc_eeftippos(self, state, offset):
        x_pose = Kinematics._forward_pos(self.model, state[:3])[0][-1]
        #apply offset of end-effector amounted at the tooltip of the links
        eef_tip = x_pose.trans + rotate(nplib.array([0, self.link_len/2, 0]) + offset, x_pose.rot)
        return eef_tip

#collect a bunch of eef tip position data
def rollout_from_ctrl(env, ctrl, offset=None):
    obs, _ = env.reset(seed=0)
    efftipobs_traj = []
    ret = 0
    if offset is None:
        eef_offset = env.offset     #this is for the real environment execution
    else:
        eef_offset = offset         #this is for the model environment executed under the hypothetical offset

    for i in range(len(ctrl)):
        obs, r, terminated, truncated, info = env.step(ctrl[i])
        eef_tip = env.calc_eeftippos(env.state, eef_offset)
        efftipobs_traj.append(eef_tip)
        ret+=r
    return ret, nplib.array(efftipobs_traj)

real_eeftip_env = MyThreeLinksPlusEndEffectorEnv()
u_traj = nplib.zeros((20, 3))
_, real_tipobs_traj = rollout_from_ctrl(real_eeftip_env, u_traj)

Let's create a model starting with the same initial condition and control. The end-effector tip positions caused by the guess of offset can be compared to the real trajectory, used in a loss function to infer the real offset:

In [8]:
model_eeftip_env = MyThreeLinksPlusEndEffectorEnv()

def calibration_loss(offset):
    _, model_tipobs_traj = rollout_from_ctrl(model_eeftip_env, u_traj, offset=offset)
    return nplib.linalg.norm(model_tipobs_traj - real_tipobs_traj)

#estimate the offset parameter, with an initial guess of 0
import scipy
result = scipy.optimize.minimize(calibration_loss, nplib.zeros(3), method="L-BFGS-B", jac='2-point', options={"maxiter":1000, "disp":False, "gtol":1e-3})
print(result)

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 2.3541376629074343e-08
        x: [-3.985e-10  1.000e-01 -3.985e-10]
      nit: 11
      jac: [-3.275e+00  1.326e+00 -3.275e+00]
     nfev: 332
     njev: 83
 hess_inv: <3x3 LbfgsInvHessProduct with dtype=float64>


As one can see the estimation successfully reveals the correct offset used in the real environment.

Inferring model parameters from real data is widely used in engineering. In particular, machines such as industrial robots often resort to [system identification](https://en.wikipedia.org/wiki/System_identification) to calibrate parameters for associated software to work properly. Machine learning relates this to the sim-to-real process and hence sometimes calls it real-to-sim for the essence of calibrating simulation with real data. This extends common system identification models when the simulation can synthesize various sensor readings and exploit automatically evaluated differentials. For instance, if we could [differentiate the rendering of a camera image](https://kaolin.readthedocs.io/en/latest/notes/diff_render.html), it will be possible to use an image-based loss by comparing pixel values directly. This is convenient when the state rollouts are not immediate to measure, e.g. when the state concerns points on deformable materials [@real2sim2022cgf].

## Interplay between Sim-to-Real and Real-to-Sim

Sim-to-Real and Real-to-Sim are complementary and thus can be jointly used when there is a need. The above example identifies a geometry parameter taken as a constant. If the target parameter is varying, e.g. the mass carried in pouring a jar of water, one may need online data collection and continual estimation to keep the model and derived control up-to-date. This creates a Real-to-Sim-to-Real loop that resembles [adaptive control](https://en.wikipedia.org/wiki/Adaptive_control) in classic automatic control [@slotine1991].   

Even if the estimation concerns invariant parameters, augmenting real data may still be beneficial. Simulation models make lots of assumptions that are not aligning with the reality perfectly so the derived actions will nonetheless underperform. Running online RL on the real system to fine-tune a policy trained in simulation may yield better performance and sometimes is necessary for successful Sim-to-Real. The online process now expects to take much less real-world interactions since a reasonably good behaviour emerges already from the simulation-based training. This is called _few-shot_ learning as the training requires a very small batch of data from the target domain. 

Real data may also benefit iterative parameter inference and domain randomization. Monitoring the real-world performance helps to identify sensitive parameter components so the simulation budget can be focused on estimating or exploring variations that are most task-relevant. One can think this as yet another level of reinforcement learning problem, for which the actions are now the parameters to experiment with while the returns are the experiment results from the real data. This is often tackled as an [active learning](https://en.wikipedia.org/wiki/Active_learning_(machine_learning)) problem [@activeDR2020Corl] with methods like [Bayesian optimization](https://en.wikipedia.org/wiki/Bayesian_optimization) [@BayesSimIG2021antonova]. 


The general software support for Sim-to-Real and the other way around requires to agglomerate multiple libraries on simulation, robot control and RL/optimization algorithms. Examples for an integrated solution can be found in [Issac Lab](https://isaac-sim.github.io/IsaacLab/main/index.html) with a rich set of built-in robots and task environments. 

